In [ ]:
import pandas as pd
import numpy as np

## Task 1: Data Ingestion

#### 1.Load all three CSV files (sales_data.csv, products.csv, stores.csv) into pandas DataFrames.Print the shape and first 5 rows of each.

In [ ]:
sales_data = pd.read_csv('sales_data.csv')
products = pd.read_csv('products.csv')
stores = pd.read_csv('stores.csv')

In [ ]:
# For sales data shape & first 5 rows:
sales_data.index = range(1,len(sales_data)+1)# start index from 1 rather then 0 make easy to understand for common users
print('Sales DataFrame shape is: ',sales_data.shape)
print('Sales DataFrame first 5 rows: \n',sales_data.head())

In [ ]:
# For products data shape & first 5 rows:
products.index = range(1,len(products)+1)# start index from 1 rather then 0 make easy to understand for common users
print('Products DataFrame shape is: ',products.shape)
print('Products DataFrame first 5 rows: \n',products.head())

In [ ]:
# For stores data shape & first 5 rows:
stores.index = range(1,len(stores)+1) # start index from 1 rather then 0 make easy to understand for common users
print('Stores DataFrame shape is: ',stores.shape)
print('Stores DataFrame first 5 rows: \n',stores.head())

#### 2. Check for missing values in all three DataFrames and print a summary of which columns have null values.

In [ ]:
# Put all data into single frame to make computation optimize
dataframe= {
    'Sales_data':sales_data,
    'Products':products,
    'Strores':stores
}

In [ ]:
for name,df in dataframe.items():
    print(f"\n\nSummary of columns with null values in : {name}")
    null_counts = df.isnull().sum()
    null_columns = null_counts[null_counts>0]
    print(null_columns)
    print(f'Total null value containing columns in {name} is: {null_columns.count()}')

## Task 2: Data Cleaning

#### 3. Remove all duplicate rows from sales_data. Print how many duplicates were found and removed.

In [ ]:
duplicate_rows = sales_data[sales_data.duplicated()]
duplicate_count = sales_data.duplicated().sum()
sales_data = sales_data.drop_duplicates()
print('Duplicate rows in sales data are: \n',duplicate_rows)
print('\n\nTotal number of duplicates found and removed in sales_data : ',duplicate_count)

#### 4. Fill missing values in the 'quantity' column with 0 and drop rows where 'amount' is NULL. Print the cleaned DataFrame shape.

In [ ]:
# print rows that contain null values in quantity column. Print to know where we are going to made changes.
print(sales_data[sales_data['quantity'].isnull()])

In [ ]:
#Fill the missing values with 0 in quantity column
sales_data['quantity'] = sales_data['quantity'].fillna(0)

In [ ]:
# To know what we are going to drop from data.
print(sales_data[sales_data['amount'].isnull()])

In [ ]:
sales_data = sales_data.dropna(subset=['amount'])

In [ ]:
print(sales_data.shape)

#### 5. Convert the 'sale_date' column to proper datetime format and the 'amount' column to float data type.

In [ ]:
# To Convert the 'sale_date' column to proper datetime format
sales_data['sale_date'] = pd.to_datetime(
    sales_data['sale_date'],
    errors='coerce')
# To convert the 'amount' column to float data type
sales_data['amount'] = pd.to_numeric(
    sales_data['amount'],
    errors='coerce')

## Task 3: Data Transformation

#### 6. Merge all three DataFrames into one final DataFrame using appropriate JOIN on store_id and product_id. Print the final merged DataFrame.

In [ ]:
# For this we use merge() as we are given where we have to join. Not join() as it works on indexes, not concat() just concat data in stack.
Final_DF = (sales_data
            .merge(stores,on = 'store_id',how='left')
            .merge(products,on ='product_id',how = 'left'))
            

In [ ]:
print(Final_DF)

#### 7. Add a new column 'total_revenue' = quantity × price. Use NumPy to calculate and print the mean, max, and min of total_revenue.

In [ ]:
# Add a new column 'total_revenue' = quantity × price
Final_DF['total_revenue'] = Final_DF['quantity']*Final_DF['price']

# Use NumPy to calculate and print the mean, max, and min of total_revenue
Mean_total_revenue = np.mean(Final_DF['total_revenue'])
Max_toal_revenue = np.max(Final_DF['total_revenue'])
Min_total_revenue = np.min(Final_DF['total_revenue'])
print(f'\n Mean of total revenue: {Mean_total_revenue}')
print(f'\n Maximum of total revenue: {Max_toal_revenue}')
print(f'\n Minimum of total revenue: {Min_total_revenue}')

#### 8. Group the data by 'city' and find the total revenue generated per city. Sort the result in descending order.

In [ ]:
Revenue_of_Cities= Final_DF.groupby('city')['total_revenue'].sum().sort_values(ascending=False)
#group on city/ take total revenue column0/ sum it on city groups/ sort the value in descending
print(Revenue_of_Cities)

## Task 4: Data Loading (SQL)

#### 9. Load the final cleaned and merged DataFrame into a SQLite database table called 'retail_sales'. Write the Python code using sqlite3 or sqlalchemy.

In [ ]:
import sqlite3 as slt

In [ ]:
# Make connection with sqllite3 and load the database in which retail_sales table is.
conn = slt.connect('retail_mart.db')

In [ ]:
# Final Dataframe is loaded into the retail_sales table in retail_mart database
Final_DF.to_sql( 
    'retail_sales',
    conn,
    if_exists='replace',
    index='false')

conn.close() # connection of python with database is closed.


#### 10. Write a SQL query to find the Top 3 best-selling products by total quantity sold from the 'retail_sales' table.

In [ ]:
# SQL query to get top 3 best selling product first add individual product total sold then group product.
# Arrange in descending order and select top 3 rows using limit.
conn = slt.connect('retail_mart.db')
query = """select product_name,sum(quantity) as total_quantity_sold 
            from retail_sales 
            group by product_name 
            order by total_quantity_sold DESC
            LIMIT 3"""
pd.read_sql(query,conn)

## Task 5: Reporting & Insights

#### 11. Write a SQL query to find total revenue per store per day from the 'retail_sales' table.

In [ ]:
# To find total revenue get sum of total_revenue as more than one product per store can be sold so it add all item revenue.
# Then group on store_name and sale_date 
revenue_query = '''select store_name,sale_date,sum(total_revenue) as Total_Revenue_per_day
                    from retail_sales 
                    group by store_name,sale_date'''
pd.read_sql(revenue_query,conn)

In [ ]:
q2 = 'select * from retail_sales'
pd.read_sql(q2,conn)

#### 12. Using Python, print a summary report showing: Total number of transactions, Total revenue, Top selling city, and Top selling product.

In [ ]:
# Total number of transaction
Total_Transactions = len(Final_DF)

# Total Revenue
Total_Revenue = Final_DF['total_revenue'].sum()

#Give name of city with highest sales
Top_selling_City = Final_DF.groupby('city')['total_revenue'].sum().idxmax()

#Give best selling product
Top_Selling_Product = Final_DF.groupby('product_name')['quantity'].sum().idxmax()

print('** ____ Summary report _____**')
print('\nTotal number of transactions: ', Total_Transactions)
print(f'\nTotal revenue: {Total_Revenue:.2f}')
print('\nCity where highest sale occured: ',Top_selling_City)
print('\nTop selling product: ',Top_Selling_Product)

## Task 6: Pipeline & Error Handling

#### 13. Write a Python function called run_pipeline() that runs all the above steps (load → clean → transform → load to DB) in one single function call.

#### 14. Add basic error handling (try-except) to your pipeline so that if any file is missing, it prints a proper error message instead of crashing.

In [ ]:
# Creating a pipeline that runs load,clean,transfom & load in single call
# Load the external CSV files for processing
def Data_ingestion():
    Sales_Data = pd.read_csv('sales_data.csv')
    Products = pd.read_csv('products.csv')
    Stores = pd.read_csv('stores.csv')
    return Sales_Data,Products,Stores

# Clean the data by removing duplicates and null values from data
def Clean_data(Sales_Data):
    Sales_Data = Sales_Data.copy()
    Sales_Data = Sales_Data.drop_duplicates()
    Sales_Data['quantity'] = Sales_Data['quantity'].fillna(0)
    Sales_Data = Sales_Data.dropna(subset=['amount'])
    return Sales_Data

#Transform the data by setting apropriate column type & merging all three CSVs in one final dataframe
def Transform_data(Sales_Data,products,stores):
    Sales_Data['sale_date'] = pd.to_datetime(
        Sales_Data['sale_date'],
        errors = 'coerce')
    
    Sales_Data['amount'] = Sales_Data['amount'].astype(float)
    
    Final_DF = (Sales_Data
                .merge(products,on='product_id',how='left')
                .merge(stores,on='store_id',how = 'left'))

    Final_DF['total_revenue'] = Final_DF['quantity']*Final_DF['price']
    
    return(Final_DF)

# Load the dataframe into sqlite3 database.
def Load_to_DB(Final_DF):
    conn = slt.connect('retail_mart.db')
    Final_DF.to_sql(
        'retail_sales',
        conn,
        if_exists = 'replace',
        index = False)
    conn.close()

# Pipeline that call all data processing steps.
def run_pipeline():
    try:
        Sales_data,Product,Stores = Data_ingestion()
        Sales_data = Clean_data(Sales_data)

        Final_df = Transform_data(Sales_data,Product,Stores)
        Load_to_DB(Final_df)
        print('Pipeline is executed successfully')
        return Final_df  
    except FileNotFoundError as NF:
        print(f'File not found: {NF}')
    except Exception as e:
        print(f'Ooh! Error :{e}')
    
                        
                 

## Final Data Insights

In [ ]:
# Insights that Business Team wants to know from all 

def Final_Insights(Final_df):
    
    print('Here is the final insights after processing the whole data')

    print('\n\n1. Which products are selling the most in which city:')
    print(Final_df.groupby(['city','product_name'])['quantity'].sum().sort_values(ascending = False))

    print('\n\n2.Total revenue generated per store per day:')
    print(Final_df.groupby(['store_name','sale_date'])['total_revenue'].sum())

    print('\n\n3.Any stores or products with missing or incorrect data: ')
    print('\nStores with null values:\n ',Final_df[Final_df['store_name'].isnull()])
    print('\nProducts with missing:\n ',Final_df[Final_df['product_name'].isnull()])
    print('\nIncorrect values in qunatity:\n',Final_df[Final_df['quantity']<0])
    print('\nIncorrect Price Values:\n',Final_df[Final_df['price']<=0])
    

In [ ]:
# Execute pipeline and give insights

import pandas as pd
import numpy as np
import sqlite3 as slt
Final_df = run_pipeline()
Final_Insights(Final_df)